### Import libraries

In [0]:
import json
import math
import re
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from delta.tables import DeltaTable
from pyspark.sql import functions as F

### Set variables

In [0]:
# Notebook parameters
# Job parameters can override these widgets.

dbutils.widgets.text("catalog", "dbw_dariusz")
dbutils.widgets.text("schema", "bronze")

dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

dbutils.widgets.text("window_days", "366")
dbutils.widgets.text("themes", "swe,sd,qtt,gwb_q,rr,tm")

dbutils.widgets.text(
    "locations_json",
    json.dumps([
        {
            "location_id": "example_utm33_130000_6800000",
            "x": 130000,
            "y": 6800000,
        }
    ]),
)

In [0]:
BASE_URL = "https://gts.nve.no/api"

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

today = datetime.now(ZoneInfo("Europe/Oslo")).date()

START_DATE = date.fromisoformat(
    dbutils.widgets.get("start_date").strip()
    or (today - timedelta(days=30)).isoformat()
)

END_DATE = date.fromisoformat(
    dbutils.widgets.get("end_date").strip()
    or (today - timedelta(days=1)).isoformat()
)

WINDOW_DAYS = int(dbutils.widgets.get("window_days"))

THEMES = [
    theme.strip()
    for theme in dbutils.widgets.get("themes").split(",")
]

LOCATIONS = json.loads(
    dbutils.widgets.get("locations_json")
)

OBSERVATION_TABLE = f"{CATALOG}.{SCHEMA}.nve_snow_hydrology"
RAW_TABLE = f"{CATALOG}.{SCHEMA}.nve_snow_hydrology_raw"

### Validate configuration

In [0]:
for identifier in (CATALOG, SCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid catalog/schema: {identifier}")

if START_DATE > END_DATE:
    raise ValueError("start_date must be <= end_date")

if END_DATE >= today:
    raise ValueError("end_date must be before today")

if not 1 <= WINDOW_DAYS <= 366:
    raise ValueError("window_days must be between 1 and 366")

if not THEMES or len(THEMES) != len(set(THEMES)):
    raise ValueError("Choose at least one theme without duplicates")

if not isinstance(LOCATIONS, list) or not LOCATIONS:
    raise ValueError("locations_json must contain a nonempty list")

location_ids = set()
coordinates = set()

for location in LOCATIONS:
    location_id = location.get("location_id")
    x = location.get("x")
    y = location.get("y")

    if (
        not isinstance(location_id, str)
        or not location_id.strip()
        or location_id in location_ids
    ):
        raise ValueError("Location IDs must be nonempty and unique")

    if type(x) is not int or type(y) is not int:
        raise ValueError("Coordinates must be integer UTM33 metres")

    if (x, y) in coordinates:
        raise ValueError("Duplicate sampling coordinates")

    location_ids.add(location_id)
    coordinates.add((x, y))

### Common functions

In [0]:
def create_session():
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})

    session.mount(
        "https://",
        HTTPAdapter(
            max_retries=Retry(
                total=4,
                backoff_factor=1,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                respect_retry_after_header=True,
            )
        ),
    )

    return session


def get_nve_data(session, path):
    response = session.get(
        f"{BASE_URL}/{path}",
        timeout=(10, 90),
    )
    response.raise_for_status()
    return response.json()


def date_windows(start, end, days):
    current = start

    while current <= end:
        stop = min(end, current + timedelta(days=days - 1))
        yield current.isoformat(), stop.isoformat()
        current = stop + timedelta(days=1)


def parse_source_time(value):
    for fmt in (
        "%d.%m.%Y %H:%M:%S",
        "%d.%m.%Y %H:%M",
        "%Y-%m-%dT%H:%M:%S",
    ):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue

    raise ValueError(f"Unsupported NVE timestamp: {value!r}")


def normalize_response(payload, location, theme, start, end):
    if payload["Theme"] != theme:
        raise ValueError("Response theme does not match request")

    if (payload["X"], payload["Y"]) != (
        location["x"],
        location["y"],
    ):
        raise ValueError("Response coordinates do not match request")

    resolution = payload["TimeResolution"]

    if resolution != 1440:
        raise ValueError("Only daily data is supported")

    first = parse_source_time(payload["StartDate"])
    last = parse_source_time(payload["EndDate"])

    if (
        first.date() != date.fromisoformat(start)
        or last.date() != date.fromisoformat(end)
    ):
        raise ValueError("Response dates do not match request")

    values = payload["Data"]
    expected_count = (
        (last - first).total_seconds() / (resolution * 60)
    ) + 1

    if not isinstance(values, list) or len(values) != expected_count:
        raise ValueError("Response contains an unexpected number of values")

    sentinel = payload["NoDataValue"]
    altitude = payload.get("Altitude")
    rows = []

    for index, raw_value in enumerate(values):
        source_time = first + timedelta(
            minutes=index * resolution
        )

        is_missing = raw_value is None or raw_value == sentinel
        value = None if is_missing else float(raw_value)

        if value is not None and not math.isfinite(value):
            raise ValueError("Response contains a non-finite value")

        rows.append({
            "location_id": location["location_id"],
            "x": location["x"],
            "y": location["y"],
            "theme": theme,
            "theme_name": payload["FullName"],
            # Preserve source wall-clock time without assuming UTC.
            "source_time": source_time.isoformat(),
            "observation_date": source_time.date(),
            "time_resolution_minutes": resolution,
            "value": value,
            "is_missing": is_missing,
            "no_data_value": float(sentinel),
            # The standard endpoint already converts to these units.
            "unit": payload["Unit"],
            "altitude_m": (
                float(altitude) if altitude is not None else None
            ),
        })

    return rows


def merge_delta(rows, schema, table, keys):
    if not rows:
        raise ValueError(f"Empty batch for {table}")

    df = (
        spark.createDataFrame(rows, schema=schema)
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("source", F.lit("NVE_GRIDTIMESERIES"))
    )

    if not spark.catalog.tableExists(table):
        (
            df.write
            .format("delta")
            .mode("error")
            .saveAsTable(table)
        )
        return

    condition = " AND ".join(
        f"target.`{key}` = incoming.`{key}`"
        for key in keys
    )

    (
        DeltaTable.forName(spark, table)
        .alias("target")
        .merge(df.alias("incoming"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

### Main code

In [0]:
OBSERVATION_SCHEMA = """
    location_id STRING,
    x INT,
    y INT,
    theme STRING,
    theme_name STRING,
    source_time STRING,
    observation_date DATE,
    time_resolution_minutes INT,
    value DOUBLE,
    is_missing BOOLEAN,
    no_data_value DOUBLE,
    unit STRING,
    altitude_m DOUBLE
"""

RAW_SCHEMA = """
    location_id STRING,
    x INT,
    y INT,
    theme STRING,
    requested_start STRING,
    requested_end STRING,
    source_url STRING,
    payload_json STRING,
    theme_metadata_json STRING
"""

In [0]:
# Fetch data and write Bronze Delta tables.
# Each request commits separately. Rerunning updates matching rows.
# Configure the Databricks job for a single concurrent run.

completed_requests = 0
observation_count = 0
missing_count = 0

with create_session() as session:
    metadata = get_nve_data(
        session,
        "GridTimeSeries/Themes/json",
    )

    available_themes = {
        item["Name"]: item
        for item in metadata
    }

    # Validate all themes before writing.
    for theme in THEMES:
        if (
            not re.fullmatch(r"[A-Za-z0-9_]+", theme)
            or theme not in available_themes
        ):
            raise ValueError(f"Unknown NVE theme: {theme}")

        info = available_themes[theme]

        if info["TimeResolutionInMinutes"] != 1440:
            raise ValueError(f"Theme is not daily: {theme}")

        first_available = date.fromisoformat(
            info["FirstDateInTimeSerie"]
        )

        if START_DATE < first_available:
            raise ValueError(
                f"{theme} is available from {first_available}"
            )

    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}"
    )

    for location in LOCATIONS:
        for theme in THEMES:
            for start, end in date_windows(
                START_DATE,
                END_DATE,
                WINDOW_DAYS,
            ):
                path = (
                    f"GridTimeSeries/"
                    f"{location['x']}/{location['y']}/"
                    f"{start}/{end}/{theme}.json"
                )

                payload = get_nve_data(session, path)

                rows = normalize_response(
                    payload,
                    location,
                    theme,
                    start,
                    end,
                )

                raw_row = {
                    "location_id": location["location_id"],
                    "x": location["x"],
                    "y": location["y"],
                    "theme": theme,
                    "requested_start": start,
                    "requested_end": end,
                    "source_url": f"{BASE_URL}/{path}",
                    "payload_json": json.dumps(
                        payload,
                        ensure_ascii=False,
                        allow_nan=False,
                    ),
                    "theme_metadata_json": json.dumps(
                        available_themes[theme],
                        ensure_ascii=False,
                    ),
                }

                merge_delta(
                    [raw_row],
                    RAW_SCHEMA,
                    RAW_TABLE,
                    [
                        "x",
                        "y",
                        "theme",
                        "requested_start",
                        "requested_end",
                    ],
                )

                merge_delta(
                    rows,
                    OBSERVATION_SCHEMA,
                    OBSERVATION_TABLE,
                    [
                        "x",
                        "y",
                        "theme",
                        "source_time",
                        "time_resolution_minutes",
                    ],
                )

                completed_requests += 1
                observation_count += len(rows)
                missing_count += sum(
                    row["is_missing"] for row in rows
                )

                print(
                    f"{location['location_id']} | {theme} | "
                    f"{start} to {end} | {len(rows)} rows"
                )

print(
    f"Completed {completed_requests} requests. "
    f"Upserted {observation_count} observations, "
    f"including {missing_count} missing values."
)

In [0]:
# Inspect the ingested date range

display(
    spark.table(OBSERVATION_TABLE)
    .filter(
        F.col("observation_date").between(
            F.lit(START_DATE),
            F.lit(END_DATE),
        )
    )
    .groupBy("location_id", "theme", "unit")
    .agg(
        F.min("observation_date").alias("first_date"),
        F.max("observation_date").alias("last_date"),
        F.count("*").alias("observation_count"),
        F.sum(
            F.col("is_missing").cast("long")
        ).alias("missing_count"),
    )
    .orderBy("location_id", "theme")
)